# Football Player Network Analysis using Apache Spark

Professional notebook structure.

In [1]:
!pip install graphframes

In [2]:
from pyspark.sql import SparkSession

sparkSession = (
    SparkSession.builder
    .appName("FootballPlaystyleAnalysis")
    .getOrCreate()
)
spark = sparkSession
sparkSession

Veri Setlerini Okuma

In [3]:
events = spark.read.option("multiline", "true").json("statsbomb-open-data/events/*.json")

In [4]:
##Events kontrolleri
events.printSchema()

events.count()

len(events.columns)

events.show(5, truncate=False)

root
 |-- 50_50: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- bad_behaviour: struct (nullable = true)
 |    |-- card: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_receipt: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_recovery: struct (nullable = true)
 |    |-- offensive: boolean (nullable = true)
 |    |-- recovery_failure: boolean (nullable = true)
 |-- block: struct (nullable = true)
 |    |-- deflection: boolean (nullable = true)
 |    |-- offensive: boolean (nullable = true)
 |    |-- save_block: boolean (nullable = true)
 |-- carry: struct (nullable = true)
 |    |-- end_location: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- clearan

In [5]:
##event types
from pyspark.sql.functions import col

events.select(col("type.name").alias("event_type")) \
      .distinct() \
      .orderBy("event_type") \
      .show(100, truncate=False)

+-----------------+
|event_type       |
+-----------------+
|50/50            |
|Bad Behaviour    |
|Ball Receipt*    |
|Ball Recovery    |
|Block            |
|Camera On        |
|Camera off       |
|Carry            |
|Clearance        |
|Dispossessed     |
|Dribble          |
|Dribbled Past    |
|Duel             |
|Error            |
|Foul Committed   |
|Foul Won         |
|Goal Keeper      |
|Half End         |
|Half Start       |
|Injury Stoppage  |
|Interception     |
|Miscontrol       |
|Offside          |
|Own Goal Against |
|Own Goal For     |
|Pass             |
|Player Off       |
|Player On        |
|Pressure         |
|Referee Ball-Drop|
|Shield           |
|Shot             |
|Starting XI      |
|Substitution     |
|Tactical Shift   |
+-----------------+



In [6]:
events.groupBy(
    col("type.name").alias("event_type")
).count().orderBy(col("count").desc()).show(100, truncate=False)

+-----------------+-------+
|event_type       |count  |
+-----------------+-------+
|Pass             |3836550|
|Ball Receipt*    |3589099|
|Carry            |3007953|
|Pressure         |1301279|
|Ball Recovery    |433698 |
|Duel             |287707 |
|Clearance        |183434 |
|Block            |157782 |
|Dribble          |138745 |
|Goal Keeper      |123920 |
|Miscontrol       |118271 |
|Foul Committed   |108845 |
|Dispossessed     |103657 |
|Foul Won         |103354 |
|Shot             |101227 |
|Interception     |90879  |
|Dribbled Past    |85864  |
|Substitution     |26481  |
|Injury Stoppage  |17200  |
|Half End         |16120  |
|Half Start       |16120  |
|50/50            |14549  |
|Tactical Shift   |10865  |
|Starting XI      |7922   |
|Referee Ball-Drop|5977   |
|Shield           |5578   |
|Player Off       |4271   |
|Player On        |4230   |
|Bad Behaviour    |2816   |
|Camera On        |2571   |
|Error            |2158   |
|Offside          |1400   |
|Camera off       |6

In [7]:
passes = events.filter(col("type.name") == "Pass")
passes.count()

3836550

In [8]:
## Extract the necessary columns for analysis
passes = passes.select(
    col("player.id").alias("passer_id"),
    col("player.name").alias("passer"),
    col("pass.recipient.id").alias("receiver_id"),
    col("pass.recipient.name").alias("receiver"),
    col("team.id").alias("team_id"),
    col("team.name").alias("team"),
    "minute",
    "second",
    "location",
    col("pass.end_location").alias("end_location")
)

##Show
passes.show(10, truncate=False)

+---------+-------------------------+-----------+-------------------------+-------+------+------+------+------------+------------+
|passer_id|passer                   |receiver_id|receiver                 |team_id|team  |minute|second|location    |end_location|
+---------+-------------------------+-----------+-------------------------+-------+------+------+------+------------+------------+
|5487     |Antoine Griezmann        |10481      |Aurélien Djani Tchouaméni|771    |France|0     |0     |[60.0, 40.0]|[48.4, 38.1]|
|10481    |Aurélien Djani Tchouaméni|24778      |Eduardo Camavinga        |771    |France|0     |2     |[47.9, 37.4]|[49.3, 28.7]|
|24778    |Eduardo Camavinga        |8519       |Dayotchanculle Upamecano |771    |France|0     |4     |[49.0, 25.2]|[38.1, 46.8]|
|8519     |Dayotchanculle Upamecano |3961       |N'Golo Kanté             |771    |France|0     |7     |[41.6, 49.4]|[49.5, 52.3]|
|3961     |N'Golo Kanté             |17592      |William Saliba           |771    |

In [9]:
passes.filter(col("receiver").isNull()).count()

245236

In [10]:
##Filtreyi uygula - alıcısı olmayan pasları çıkar
passes = passes.filter(col("receiver").isNotNull())
## Sayı kontrol
passes.count()

3591314

In [11]:
## create an edge list for the passes
edges = (
    passes.groupBy(
        "passer_id",
        "passer",
        "receiver_id",
        "receiver",
        "team_id",
        "team"
    )
    .count()
    .withColumnRenamed("count", "pass_count")

    
    
)

##kontrol
edges.show(20, truncate=False)

edges.count()

+---------+-----------------------------+-----------+------------------------------+-------+---------------+----------+
|passer_id|passer                       |receiver_id|receiver                      |team_id|team           |pass_count|
+---------+-----------------------------+-----------+------------------------------+-------+---------------+----------+
|8519     |Dayotchanculle Upamecano     |4445       |Jules Koundé                  |771    |France         |76        |
|5204     |Bruno Miguel Borges Fernandes|41092      |Nuno Mendes                   |780    |Portugal       |39        |
|3961     |N'Golo Kanté                 |3009       |Kylian Mbappé Lottin          |771    |France         |59        |
|10595    |Raphael Dias Belloli         |3063       |Danilo Luiz da Silva          |781    |Brazil         |20        |
|30486    |Pedro González López         |5203       |Sergio Busquets i Burgos      |772    |Spain          |60        |
|16022    |Christian Fassnacht          

205188

In [12]:
## creating verteices

vertices = (
    passes.select(
        col("passer_id").alias("id"),
        col("passer").alias("name"),
        "team_id",
        "team"
    )
    .distinct()
)

vertices.count()
vertices.show(20, truncate=False)

+-----+--------------------------------+-------+------------------------+
|id   |name                            |team_id|team                    |
+-----+--------------------------------+-------+------------------------+
|3009 |Kylian Mbappé Lottin            |131    |Paris Saint-Germain     |
|6840 |Marcos Llorente Moreno          |772    |Spain                   |
|23725|Roman Bezus                     |911    |Ukraine                 |
|5487 |Antoine Griezmann               |212    |Atlético Madrid         |
|34639|Vitor Machado Ferreira          |131    |Paris Saint-Germain     |
|48396|Rocco Reitz                     |185    |Borussia Mönchengladbach|
|13620|Éder Gabriel Militão            |781    |Brazil                  |
|25305|Pedro Guilherme Abreu dos Santos|781    |Brazil                  |
|18618|Serhiy Kryvtsov                 |911    |Ukraine                 |
|31900|Oleksandr Karavaev              |911    |Ukraine                 |
|11396|Florian Grillitsch             

Matches'i okuma işlemi

In [1]:
matches = spark.read.option(
    "multiline", "true"
).json("statsbomb-open-data/matches/*/*.json")

NameError: name 'spark' is not defined

In [ ]:
## Match kontrolleri

matches.select(
    "match_id",
    "competition.competition_name",
    "season.season_name",
    "home_team.home_team_name",
    "away_team.away_team_name",
    "match_date"
).show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import input_file_name

events = spark.read.option(
    "multiline", "true"
).json("statsbomb-open-data/events/*.json") \
.withColumn("file", input_file_name())

In [ ]:
##vertex ve edge sayısı
vertices.count(), edges.count()

In [ ]:
## en çok pas alan ve atan oyuncular
from pyspark.sql.functions import sum

edges.groupBy("passer") \
     .agg(sum("pass_count").alias("total_passes")) \
     .orderBy(col("total_passes").desc()) \
     .show(20, truncate=False)

edges.groupBy("receiver") \
     .agg(sum("pass_count").alias("received_passes")) \
     .orderBy(col("received_passes").desc()) \
     .show(20, truncate=False)

In [ ]:
## en fazla pas yapan ikililer
edges.orderBy(col("pass_count").desc()).show(20, truncate=False)

In [ ]:
events.inputFiles()[:5]

## Events match_id ile yeniden oku


In [ ]:
from pyspark.sql.functions import input_file_name, regexp_extract

events = (
    spark.read
    .option("multiline", True)
    .json("statsbomb-open-data/events/*.json")
    .withColumn("file_path", input_file_name())
    .withColumn(
        "match_id",
        regexp_extract("file_path", r"(\d+)\.json$", 1).cast("int")
    )
)

In [ ]:
events.select("match_id").show(10)

In [ ]:
events.select("match_id").distinct().count()

In [ ]:
matches.select("match_id").distinct().count()

numbers match

In [ ]:
## Reconstruct passes dataframe with match_id

from pyspark.sql.functions import col

passes = (
    events
    .filter(col("type.name") == "Pass")
    .select(
        "match_id",
        col("player.id").alias("passer_id"),
        col("player.name").alias("passer"),
        col("pass.recipient.id").alias("receiver_id"),
        col("pass.recipient.name").alias("receiver"),
        col("team.id").alias("team_id"),
        col("team.name").alias("team"),
        "minute",
        "second"
    )
    .filter(col("receiver").isNotNull())
    .cache()
)

passes.count()   # cache'i doldurur
# Temizlenmiş ve düzleştirilmiş pas verisini Parquet formatında diske kaydet
# partitionBy ile veriyi maçlara göre klasörleyerek bölümlüyoruz (Optimization)
(
    passes.write
    .mode("overwrite")
    .partitionBy("match_id")
    .parquet("statsbomb-open-data/data/processed/passes")
)
print("Pas verileri başarıyla diske kaydedildi!")

In [ ]:
passes.printSchema()

In [ ]:
## Reconstruct edges dataframe with match_id
edges = (
    passes.groupBy(
        "match_id",
        "passer_id",
        "passer",
        "receiver_id",
        "receiver",
        "team_id",
        "team"
    )
    .count()
    .withColumnRenamed("count", "pass_count")
)

In [ ]:
## Graph Validation

# Kaç oyuncu?
vertices.count()

# Kaç edge?
edges.count()

# Kaç maç?
edges.select("match_id").distinct().count()

In [ ]:
## En fazla pas yapan ikililer
edges.orderBy(
    col("pass_count").desc()
).show(20, truncate=False)

## Graph Analytics

In [ ]:
from pyspark.sql.functions.builtin import countDistinct

out_degree = (
    edges
    .groupBy("passer_id", "passer")
    .agg(
        countDistinct("receiver_id").alias("out_degree")
    )
)

out_degree.orderBy("out_degree", ascending=False).show(20, truncate=False)

In [ ]:
##In-Degree Analysis
in_degree = (
    edges
    .groupBy("receiver_id", "receiver")
    .agg(
        countDistinct("passer_id").alias("in_degree")
    )
)

in_degree.orderBy("in_degree", ascending=False).show(20, truncate=False)

In [ ]:
## TOTAL DEGREE
from pyspark.sql.functions import coalesce, col

degree = (
    out_degree.alias("o")
    .join(
        in_degree.alias("i"),
        col("o.passer_id") == col("i.receiver_id"),
        "full"
    )
    .select(
        coalesce(col("o.passer_id"), col("i.receiver_id")).alias("player_id"),
        coalesce(col("o.passer"), col("i.receiver")).alias("player"),
        coalesce(col("out_degree"), col("in_degree") * 0).alias("out_degree"),
        coalesce(col("in_degree"), col("out_degree") * 0).alias("in_degree")
    )
    .withColumn(
        "degree",
        col("out_degree") + col("in_degree")
    )
)

degree.orderBy(
    col("degree").desc()
).show(20, truncate=False)

## NETWORK DENSITY

In [ ]:
# NETWORK DENSITY

num_vertices = vertices.count()
num_edges = edges.count()

density = num_edges / (num_vertices * (num_vertices - 1))

print(f"Number of Players (Vertices): {num_vertices}")
print(f"Number of Passing Connections (Edges): {num_edges}")
print(f"Network Density: {density:.6f}")

## Network Density

Network density measures how many passing connections exist compared to the maximum possible number of connections in a directed network.

The density is calculated as:

Density = |E| / (|V| × (|V| − 1))

where:
- |E| is the number of edges (passing connections)
- |V| is the number of vertices (players)

A low density indicates that only a small fraction of all possible passing relationships actually occur, which is expected in football passing networks since players usually interact with only a subset of teammates.

In [ ]:
## TOTAL PASSES BY TEAM
from pyspark.sql.functions import sum as spark_sum, col

team_passes = (
    edges
    .groupBy("team")
    .agg(
        spark_sum("pass_count").alias("total_passes")
    )
    .orderBy(col("total_passes").desc())
)

team_passes.show(truncate=False)

In [ ]:
## NUMBER OF PLAYERS PER TEAM
team_players = (
    vertices
    .groupBy("team")
    .count()
    .withColumnRenamed("count", "number_of_players")
    .orderBy(col("number_of_players").desc())
)

team_players.show(truncate=False)

In [ ]:
vertices.count()
vertices.select("id").distinct().count()

In [ ]:
vertices.filter(col("team") == "Barcelona").show(20, truncate=False)

In [ ]:
## AVG PASSES PER PLAYER
team_statistics = (
    team_passes.alias("p")
    .join(
        team_players.alias("t"),
        "team"
    )
    .withColumn(
        "avg_passes_per_player",
        col("total_passes") / col("number_of_players")
    )
    .orderBy(col("total_passes").desc())
)

team_statistics.show(truncate=False)

In [ ]:
## top 10 team by passing activity
top10_teams = (
    team_passes
    .limit(10)
)

top10_teams.show(truncate=False)

In [ ]:
##TOTAL SUCCESSFUL PASSES BY TEAM
from pyspark.sql.functions import sum as spark_sum, col

team_passes = (
    edges
    .groupBy("team")
    .agg(
        spark_sum("pass_count").alias("total_successful_passes")
    )
    .orderBy(col("total_successful_passes").desc())
)

team_passes.show(truncate=False)

In [ ]:
## avg passes per match by team
from pyspark.sql.functions import countDistinct

team_avg_passes = (
    edges
    .groupBy("team")
    .agg(
        spark_sum("pass_count").alias("total_passes"),
        countDistinct("match_id").alias("matches_played")
    )
    .withColumn(
        "average_passes_per_match",
        col("total_passes") / col("matches_played")
    )
    .orderBy(col("average_passes_per_match").desc())
)

team_avg_passes.show(truncate=False)

## Team Statistics

This section summarizes the passing activity of each team.

The following metrics are computed:

- Total successful passes by team
- Average successful passes per match
- Top 10 teams ranked by passing activity

These statistics provide an overview of team playing styles and overall passing performance across the dataset.

## Match Stats

In [ ]:
## Read matches dataset

matches_df = (
    spark.read
    .option("multiline", "true")
    .json("statsbomb-open-data/matches/*/*.json")
)

print(f"Total number of matches: {matches_df.count()}")

matches_df.printSchema()

In [ ]:
from pyspark.sql import functions as F

matches = matches_df.select(
    F.col("match_id"),
    F.col("match_date"),
    F.col("competition.competition_name").alias("competition"),
    F.col("season.season_name").alias("season"),
    F.col("home_team.home_team_name").alias("home_team"),
    F.col("away_team.away_team_name").alias("away_team"),
    F.col("home_score"),
    F.col("away_score")
)

matches.show(5, truncate=False)

In [ ]:
match_passes = (
    edges.join(matches, on="match_id", how="left")
)

match_passes.show(5, truncate=False)

In [ ]:
## En fazla pas yapılan maçlar

from pyspark.sql import functions as F

match_total_passes = (
    match_passes
    .groupBy(
        "match_id",
        "match_date",
        "competition",
        "season",
        "home_team",
        "away_team"
    )
    .agg(
        F.sum("pass_count").alias("total_passes")
    )
    .orderBy(F.desc("total_passes"))
)

match_total_passes.show(10, truncate=False)

In [ ]:
match_total_passes.describe().show()

In [ ]:
## Takımların maç başına pas sayısı
team_match_passes = (
    match_passes
    .groupBy(
        "match_id",
        "match_date",
        "competition",
        "home_team",
        "away_team",
        "team"
    )
    .agg(
        F.sum("pass_count").alias("passes")
    )
    .orderBy(F.desc("passes"))
)

team_match_passes.show(20, truncate=False)

Observation: Teams with higher total pass counts generally tend to adopt a possession-oriented style of play, emphasizing ball retention, build-up play, and controlled attacks. In contrast, teams with lower pass counts may rely on a more direct or defensive approach, often focusing on compact defending and quick counterattacks rather than prolonged possession. However, passing volume alone is not sufficient to classify a team's tactical approach, as match context, opponent quality, and game state can significantly influence passing behavior.

In [ ]:
## KAÇ FARKLI PASSER -> RECEIVER İKİLİSİ VAR?
unique_connections = (
    match_passes
    .groupBy(
        "match_id",
        "match_date",
        "home_team",
        "away_team"
    )
    .agg(
        F.count("*").alias("unique_passing_connections")
    )
    .orderBy(F.desc("unique_passing_connections"))
)

unique_connections.show(20, truncate=False)

In [ ]:
## Her maçta pas yapan oyuncu sayısı
players_per_match = (
    match_passes
    .groupBy(
        "match_id",
        "match_date",
        "home_team",
        "away_team"
    )
    .agg(
        F.countDistinct("passer_id").alias("players_involved")
    )
    .orderBy(F.desc("players_involved"))
)

players_per_match.show(20, truncate=False)

In [ ]:
## Maç başına istatistikler - min max. avg vs
match_summary = (
    match_total_passes
    .select("total_passes")
    .summary(
        "count",
        "mean",
        "stddev",
        "min",
        "max"
    )
)

match_summary.show()

## PLAYER STATS

In [ ]:
## En çok pas atan oyuncular
from pyspark.sql import functions as F

top_passers = (
    edges
    .groupBy("passer_id", "passer")
    .agg(
        F.sum("pass_count").alias("total_passes")
    )
    .orderBy(F.desc("total_passes"))
)

top_passers.show(20, truncate=False)

In [ ]:
## En çok pas alan oyuncular
top_receivers = (
    edges
    .groupBy("receiver_id", "receiver")
    .agg(
        F.sum("pass_count").alias("received_passes")
    )
    .orderBy(F.desc("received_passes"))
)
top_receivers.show(20, truncate=False)

In [ ]:
## Oyuncu katılımı (pas alma + pas verme)
sent = (
    edges
    .groupBy("passer_id", "passer")
    .agg(F.sum("pass_count").alias("passes_sent"))
)

received = (
    edges
    .groupBy("receiver_id", "receiver")
    .agg(F.sum("pass_count").alias("passes_received"))
    .withColumnRenamed("receiver_id", "player_id")
    .withColumnRenamed("receiver", "player")
)

sent = (
    sent
    .withColumnRenamed("passer_id", "player_id")
    .withColumnRenamed("passer", "player")
)

player_involvement = (
    sent.join(received, ["player_id", "player"], "outer")
    .fillna(0)
    .withColumn(
        "total_involvement",
        F.col("passes_sent") + F.col("passes_received")
    )
    .orderBy(F.desc("total_involvement"))
)

player_involvement.show(20, truncate=False)

In [ ]:
## En güçlü oyuncu ikilileri
top_pairs = (
    edges
    .groupBy(
        "passer",
        "receiver"
    )
    .agg(
        F.sum("pass_count").alias("passes_between")
    )
    .orderBy(F.desc("passes_between"))
)

top_pairs.show(20, truncate=False)

In [ ]:
## En fazla farklı oyuncu ile bağlantı kuranlar
most_connections = (
    edges
    .groupBy(
        "passer_id",
        "passer"
    )
    .agg(
        F.countDistinct("receiver_id").alias("unique_teammates")
    )
    .orderBy(F.desc("unique_teammates"))
)

most_connections.show(20, truncate=False)

In [ ]:
## En fazla farklı oyuncudan pas alanlar
most_received_connections = (
    edges
    .groupBy(
        "receiver_id",
        "receiver"
    )
    .agg(
        F.countDistinct("passer_id").alias("unique_passers")
    )
    .orderBy(F.desc("unique_passers"))
)

most_received_connections.show(20, truncate=False)

In [ ]:
top_passers.select("total_passes").summary(
    "count", ##oyıuncu sayısı
    "mean", ## oyuncu başına ortalama pas sayısı
    "stddev", ## standart sapma
    "min", ## en az pas atan oyuncu
    "max" ## en çok pas atan oyuncu
).show()

## GraphFrames ile Graph Kurulumu ve PageRankBu bölüme kadar oyuncu derecesi (in/out degree) ve pas istatistikleri manuel DataFrame join/groupBy işlemleriyle hesaplandı. Bu bölümde ise pas ağı, GraphFrames kütüphanesi kullanılarak bir graph nesnesine dönüştürülüp PageRank algoritması ile analiz edilecek.**Not:** GraphFrames'in çalışması için Spark session'ın GraphFrames JAR paketiyle başlatılmış olması gerekir. Eğer bu notebook'u baştan çalıştırıyorsanız, ilk hücredeki SparkSession oluşturma kodunu aşağıdaki gibi güncelleyin:```pythonsparkSession = (    SparkSession.builder    .appName("FootballPlaystyleAnalysis")    .master("local[*]")    .config("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.5-s_2.12")    .getOrCreate())```Ayrıca `pip install graphframes` ile Python tarafındaki wrapper'ın da kurulu olması gerekiyor.

In [2]:
## GraphFrames import
from graphframes import GraphFrame
from pyspark.sql import functions as F
#%%
## Vertices: GraphFrame "id" kolonu bekliyor, elimizdeki vertices DataFrame'inde zaten var
gf_vertices = vertices.select("id", "name", "team_id", "team").distinct()

gf_vertices.show(10, truncate=False)
gf_vertices.count()
#%%
## Edges: GraphFrame "src" ve "dst" kolonlarını bekliyor.
## Maçlardan bağımsız, oyuncu bazlı toplam pas sayısına göre global bir graph kuruyoruz.
gf_edges = (
    edges
    .groupBy("passer_id", "receiver_id")
    .agg(F.sum("pass_count").alias("weight"))
    .withColumnRenamed("passer_id", "src")
    .withColumnRenamed("receiver_id", "dst")
)

gf_edges.show(10, truncate=False)
gf_edges.count()

# Graf düğümlerini (oyuncular) kaydet
gf_vertices.write.mode("overwrite").parquet("statsbomb-open-data/data/processed/vertices")

# Graf kenarlarını (pas bağlantıları) kaydet
gf_edges.write.mode("overwrite").parquet("statsbomb-open-data/data/processed/edges")

NameError: name 'vertices' is not defined

In [ ]:
## Graph nesnesinin oluşturulması
g = GraphFrame(gf_vertices, gf_edges)

print(f"Graph vertices: {g.vertices.count()}")
print(f"Graph edges: {g.edges.count()}")

# PageRank hesaplaması
pr_results = g.pageRank(resetProbability=0.15, maxIter=10)

# En yüksek PageRank skoruna sahip ilk 20 oyuncuyu görelim
print("--- En Etkili Oyun Kurucular (PageRank) ---")
pr_results.vertices.orderBy("pagerank", ascending=False).show(20, truncate=False)

In [ ]:
# Düğümlerin dahil olduğu üçgen (triangle) sayısını hesapla
triangles = g.triangleCount()

print("--- En Çok Pas Üçgeni Kuran Oyuncular ---")
triangles.orderBy("count", ascending=False).show(20, truncate=False)

In [ ]:
print("Spark version:", spark.version)
print("Scala version:", spark.sparkContext._jvm.scala.util.Properties.versionString())

In [ ]:
#Flag diske kaydettiğimiz verileri okuyup kaldığımız yerden devam edebilmek için
passes = spark.read.parquet("statsbomb-open-data/data/processed/passes")
gf_vertices = spark.read.parquet("statsbomb-open-data/data/processed/vertices")
gf_edges = spark.read.parquet("statsbomb-open-data/data/processed/edges")